# 🤖 Model Training — Baseline ML + BERT
Train and evaluate all sentiment models.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
# ── Load cleaned data ──────────────────────────────────────────────────────
CLEAN_PATH = '../data/cleaned_reviews.csv'

if not os.path.exists(CLEAN_PATH):
    from scraper.scraper import generate_sample_reviews
    from preprocessing.preprocess import preprocess_dataframe
    raw = pd.DataFrame(generate_sample_reviews(500))
    df = preprocess_dataframe(raw)
    os.makedirs('../data', exist_ok=True)
    df.to_csv(CLEAN_PATH, index=False)
else:
    df = pd.read_csv(CLEAN_PATH)

print(f'Dataset: {df.shape}')
print(df['sentiment'].value_counts())

In [ ]:
# ── Train all baseline models ──────────────────────────────────────────────
from models.baseline_model import train_all, save_model

results = train_all(df)

# Summary table
summary = pd.DataFrame([
    {'Model': k,
     'Test Accuracy': v['accuracy'],
     'CV Mean': v['cv_mean'],
     'CV Std': v['cv_std']}
    for k, v in results.items()
])
print(summary.sort_values('CV Mean', ascending=False).to_string(index=False))

In [ ]:
# ── Plot model comparison ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
models_names = summary['Model'].tolist()
x = np.arange(len(models_names))
width = 0.35

bars1 = ax.bar(x - width/2, summary['Test Accuracy'], width,
               label='Test Accuracy', color='#6366f1', alpha=0.85)
bars2 = ax.bar(x + width/2, summary['CV Mean'], width,
               label='CV Mean', color='#22c55e', alpha=0.85,
               yerr=summary['CV Std'], capsize=4)

ax.set_xticks(x)
ax.set_xticklabels([m.replace('_', ' ').title() for m in models_names])
ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy')
ax.set_title('Baseline Model Comparison', fontsize=14, fontweight='bold')
ax.legend()
ax.bar_label(bars1, fmt='%.3f', padding=3, fontsize=9)
ax.bar_label(bars2, fmt='%.3f', padding=3, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion matrices ─────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

X = df['cleaned_text'].values
y = df['label'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                     random_state=42, stratify=y)
labels = ['negative', 'neutral', 'positive']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, result) in zip(axes, results.items()):
    cm = result['confusion_matrix']
    disp = ConfusionMatrixDisplay(confusion_matrix=np.array(cm), display_labels=labels)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name.replace('_', ' ').title(), fontsize=12, fontweight='bold')

plt.suptitle('Confusion Matrices', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Save best model ────────────────────────────────────────────────────────
best_name = summary.sort_values('CV Mean', ascending=False).iloc[0]['Model']
best_pipeline = results[best_name]['model']
save_model(best_pipeline, '../models/baseline_model.pkl')
print(f'Best model saved: {best_name}')

In [ ]:
# ── Optional: Fine-tune BERT ───────────────────────────────────────────────
# Uncomment and run on a GPU machine / Colab for best results.

# from models.bert_model import BERTSentimentTrainer
# trainer = BERTSentimentTrainer(epochs=3, batch_size=16)
# trainer.train(df)
print('BERT training cell — uncomment to run on GPU.')

In [ ]:
# ── TF-IDF feature importance (top words per class) ────────────────────────
import pickle
from sklearn.linear_model import LogisticRegression

lr_result = results.get('logistic_regression')
if lr_result:
    pipe = lr_result['model']
    tfidf = pipe.named_steps['tfidf']
    clf = pipe.named_steps['clf']
    feature_names = tfidf.get_feature_names_out()

    fig, axes = plt.subplots(1, 3, figsize=(18, 7))
    class_labels = ['negative', 'neutral', 'positive']
    palette = ['#ef4444', '#f59e0b', '#22c55e']

    for i, (ax, label, color) in enumerate(zip(axes, class_labels, palette)):
        coef = clf.coef_[i]
        top_idx = np.argsort(coef)[-20:][::-1]
        top_words = feature_names[top_idx]
        top_coef = coef[top_idx]
        ax.barh(top_words[::-1], top_coef[::-1], color=color, alpha=0.85)
        ax.set_title(f'Top Features — {label.title()}', fontsize=12, fontweight='bold')
        ax.set_xlabel('Coefficient')

    plt.suptitle('Logistic Regression Feature Importance', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()